In [1]:
account_path = '/content/drive/MyDrive/' # primary lab account
# account_path = '/content/drive/MyDrive/Personal_Notebook/Johsien/' # backup account

# Mount & Import

In [2]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive/')
from google.colab import auth
auth.authenticate_user()

# load packages - basics
!pip install pynrrd
import os
import csv
import numpy as np
import pandas as pd
import math
import time
import json
import nrrd

Mounted at /content/drive/


In [3]:
# load packages - stats
!pip install scikit-posthocs
import scipy
from scipy import io
from scipy.fftpack import rfft, irfft, fftfreq
import scipy.stats as stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
import scikit_posthocs as scikit
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels

import random
from random import randrange, shuffle
import itertools
from itertools import combinations
import sklearn
from sklearn.decomposition import PCA, FastICA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score

In [4]:
# load packages - plotting and template settings
import matplotlib as mpl
from matplotlib import pyplot as plt
%matplotlib inline

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import animation
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [5]:
# load custom functions and sleap skeleton
%cd '/content/drive/MyDrive/Yu_escape_paper/Colab_github/src'
%run jy_utils.ipynb
%run helper_funcs_SE.ipynb
sleap_nodes = ['le','re','fb','mb','rb','t1','t2','t3','tt'] # sleap_nodes_jy

/content/drive/MyDrive/Yu_escape_paper/Colab_github/src


# Experimental list (SE)

In [6]:
gsheet = 'https://docs.google.com/spreadsheets/d/1bSfcczIO4V62vPDB93Xjm2VDwlayalswnmi6PDiiLps/edit?gid=0#gid=0'
tab_name = 'SE'

experiment_class = f'Social escape - {tab_name}'
path = account_path + 'Yu_escape_paper/Behavior/' + experiment_class

data_path = path + '/Data/'
save_path = path + '/Analysis/'

metadata = read_metadata(gsheet,tab_name)
experiments = metadata.index

,experiment,time_stamps,fish_num,sex,exp_type,fps_Hz,rig_mm,vid_px,body_length,experiment_date,condition
0,vid_1.mp4.000_vid_1.analysis.h5,vid_1.csv,1,M,Panda3D_DC_SE,40,265,800,12,22025,3_trials
1,vid_2.mp4.000_vid_2.analysis.h5,vid_2.csv,1,M,Panda3D_DC_SE,40,265,800,12,22025,3_trials
2,vid_3.mp4.000_vid_3.analysis.h5,vid_3.csv,1,M,Panda3D_DC_SE,40,265,800,12,22025,3_trials
3,vid_4.mp4.000_vid_4.analysis.h5,vid_4.csv,1,M,Panda3D_DC_SE,40,265,800,12,22025,3_trials
4,vid_5.mp4.000_vid_5.analysis.h5,vid_5.csv,1,M,Panda3D_DC_SE,40,265,800,12,22025,3_trials
5,vid_6.mp4.000_vid_6.analysis.h5,vid_6.csv,1,M,Panda3D_DC_SE,40,265,800,12,22025,3_trials
6,vid_8.mp4.000_vid_8.analysis.h5,vid_8.csv,1,F,Panda3D_DC_SE,40,265,800,12,81125,3_trials
7,vid_7.mp4.000_vid_7.analysis.h5,vid_7.csv,1,F,Panda3D_DC_SE,40,265,800,12,81125,3_trials
8,vid_6.mp4.000_vid_6.analysis.h5,vid_6.csv,1,M,Panda3D_DC_SE,40,265,800,12,81125,3_trials
9,vid_5.mp4.000_vid_5.analysis.h5,vid_5.csv,1,M,Panda3D_DC_SE,40,265,800,12,81125,3_trials


# process

In [ ]:
for exp in experiments:
  if type(metadata.loc[exp]) ==pd.Series: # only 1 unique index
    load_path = data_path + f'{metadata.experiment_date[exp]}/'
    # print(load_path)
    output_path = save_path + f'{metadata.experiment_date[exp]}/'
    # print(output_path)

    os.makedirs(output_path, exist_ok=True)

    print(exp)
    exp_name = exp.split('.')[0]

    # load metadata
    exp_type = metadata.exp_type[exp]
    fish_num = metadata.fish_num[exp]
    fps_Hz = metadata.fps_Hz[exp]
    rig_mm = metadata.rig_mm[exp]
    vid_px = metadata.vid_px[exp]
    condition = metadata.condition[exp]
    scale = rig_mm / vid_px # mm/pixel

    # get arena dimensions
    vid_name = exp.split('.')[0]; print(vid_name)
    BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
    idx = get_BBOX_IDX(BoxDims,vid_name)
    ArenaDims = BoxDims.iloc[idx,:].to_dict()

    # load the h5
    h5_obj = h5py.File(load_path + exp, "r")

    h5tracks = h5_obj['tracks'][:].T
    frame_num, _, _, track_num = h5tracks.shape
    print("frames x nodes x 2 x animals", h5tracks.shape)

    # load data from h5 tracks
    fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

    # get time stamps
    time_stamps = getTimeStamps(load_path + vid_name + '.csv')

    # plot all trajectories
    plt.figure(figsize=[3,3])
    for i in fish_data:
      byx = i.__getattribute__('mbx')
      byy = i.__getattribute__('mby')
      plt.scatter(byx,byy, s=3, alpha=0.1)

    # # organize them into group arrays and save as lev0_basics.npz
    f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
    os.makedirs(output_path+exp, exist_ok=True)
    np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
            time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
            f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
            f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
            f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
            f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition)

    # break
  else:
    for i in range(len(metadata.loc[exp])):
      load_path = data_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(load_path)
      output_path = save_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(output_path)

      os.makedirs(output_path, exist_ok=True)

      print(exp)
      exp_name = exp.split('.')[0]

      # load metadata
      exp_type = metadata.exp_type[exp].iloc[i]
      fish_num = metadata.fish_num[exp].iloc[i]
      fps_Hz = metadata.fps_Hz[exp].iloc[i]
      rig_mm = metadata.rig_mm[exp].iloc[i]
      vid_px = metadata.vid_px[exp].iloc[i]
      condition = metadata.condition[exp].iloc[i]
      scale = rig_mm / vid_px # mm/pixel

      # get arena dimensions
      vid_name = exp.split('.')[0]; print(vid_name)
      BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
      idx = get_BBOX_IDX(BoxDims,vid_name)
      ArenaDims = BoxDims.iloc[idx,:].to_dict()

      # # get metdata for vid
      # vid_metadata = pd.read_csv(load_path+exp_name+'_metadata.csv')
      # vid_metadata = vid_metadata.set_index("Parameter")["Value"].to_dict()

      # load the h5
      h5_obj = h5py.File(load_path + exp, "r")

      h5tracks = h5_obj['tracks'][:].T
      frame_num, _, _, track_num = h5tracks.shape
      print("frames x nodes x 2 x animals", h5tracks.shape)

      # load data from h5 tracks
      fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

      # get time stamps
      time_stamps = getTimeStamps(load_path + vid_name + '.csv')

      # plot all trajectories
      plt.figure(figsize=[3,3])
      for i in fish_data:
        byx = i.__getattribute__('mbx')
        byy = i.__getattribute__('mby')
        plt.scatter(byx,byy, s=3, alpha=0.1)
        plt.show()

      # # organize them into group arrays and save as lev0_basics.npz
      f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
      os.makedirs(output_path+exp, exist_ok=True)
      np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
              time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
              f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
              f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
              f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
              f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition,vid_metadata=vid_metadata)

    # break

# Experimental list (CL)

In [7]:
gsheet = 'https://docs.google.com/spreadsheets/d/1bSfcczIO4V62vPDB93Xjm2VDwlayalswnmi6PDiiLps/edit?gid=0#gid=0'
tab_name = 'CL'

experiment_class = f'Social escape - {tab_name}'
path = account_path + 'Yu_escape_paper/Behavior/' + experiment_class

data_path = path + '/Data/'
save_path = path + '/Analysis/'

metadata = read_metadata(gsheet,tab_name)
experiments = metadata.index

,experiment,time_stamps,fish_num,sex,exp_type,fps_Hz,rig_mm,vid_px,body_length,experiment_date,condition
0,vid_1.mp4.000_vid_1.analysis.h5,vid_1.csv,1,M,Panda3D_DC_SE_CL,13,265,800,12,31225,5
1,vid_2.mp4.000_vid_2.analysis.h5,vid_2.csv,1,M,Panda3D_DC_SE_CL,13,265,800,12,31225,5
2,vid_3.mp4.000_vid_3.analysis.h5,vid_3.csv,1,M,Panda3D_DC_SE_CL,13,265,800,12,31225,5
3,vid_4.mp4.000_vid_4.analysis.h5,vid_4.csv,1,M,Panda3D_DC_SE_CL,13,265,800,12,31225,5
4,vid_5.mp4.000_vid_5.analysis.h5,vid_5.csv,1,M,Panda3D_DC_SE_CL,13,265,800,12,31225,5
5,vid_6.mp4.000_vid_6.analysis.h5,vid_6.csv,1,M,Panda3D_DC_SE_CL,13,265,800,12,31225,5
6,vid_7.mp4.000_vid_7.analysis.h5,vid_7.csv,1,F,Panda3D_DC_SE_CL,13,265,800,12,31225,5
7,vid_8.mp4.000_vid_8.analysis.h5,vid_8.csv,1,M,Panda3D_DC_SE_CL,13,265,800,12,31225,5
8,vid_9.mp4.000_vid_9.analysis.h5,vid_9.csv,1,M,Panda3D_DC_SE_CL,13,265,800,12,31225,5
9,vid_10.mp4.000_vid_10.analysis.h5,vid_10.csv,1,F,Panda3D_DC_SE_CL,13,265,800,12,31225,5


# process

In [ ]:
for exp in experiments:
  if type(metadata.loc[exp]) ==pd.Series: # only 1 unique index
    load_path = data_path + f'{metadata.experiment_date[exp]}/'
    print(load_path)
    output_path = save_path + f'{metadata.experiment_date[exp]}/'
    print(output_path)

    os.makedirs(output_path, exist_ok=True)

    print(exp)
    exp_name = exp.split('.')[0]

    # load metadata
    exp_type = metadata.exp_type[exp]
    fish_num = metadata.fish_num[exp]
    fps_Hz = metadata.fps_Hz[exp]
    rig_mm = metadata.rig_mm[exp]
    vid_px = metadata.vid_px[exp]
    condition = metadata.condition[exp]
    scale = rig_mm / vid_px # mm/pixel

    # get arena dimensions
    vid_name = exp.split('.')[0]; print(vid_name)
    BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
    idx = get_BBOX_IDX(BoxDims,vid_name)
    ArenaDims = BoxDims.iloc[idx,:].to_dict()

    # load the h5
    h5_obj = h5py.File(load_path + exp, "r")

    h5tracks = h5_obj['tracks'][:].T
    frame_num, _, _, track_num = h5tracks.shape
    print("frames x nodes x 2 x animals", h5tracks.shape)

    # load data from h5 tracks
    fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

    # get time stamps
    time_stamps = getTimeStamps(load_path + vid_name + '.csv')

    # plot all trajectories
    plt.figure(figsize=[3,3])
    for i in fish_data:
      byx = i.__getattribute__('mbx')
      byy = i.__getattribute__('mby')
      plt.scatter(byx,byy, s=3, alpha=0.1)
      plt.show()

    # # organize them into group arrays and save as lev0_basics.npz
    f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
    os.makedirs(output_path+exp, exist_ok=True)
    np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
            time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
            f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
            f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
            f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
            f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition)

  else:
    for i in range(len(metadata.loc[exp])):
      load_path = data_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(load_path)
      output_path = save_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(output_path)

      os.makedirs(output_path, exist_ok=True)

      print(exp)
      exp_name = exp.split('.')[0]

      # load metadata
      exp_type = metadata.exp_type[exp].iloc[i]
      fish_num = metadata.fish_num[exp].iloc[i]
      fps_Hz = metadata.fps_Hz[exp].iloc[i]
      rig_mm = metadata.rig_mm[exp].iloc[i]
      vid_px = metadata.vid_px[exp].iloc[i]
      condition = metadata.condition[exp].iloc[i]
      scale = rig_mm / vid_px # mm/pixel

      # get arena dimensions
      vid_name = exp.split('.')[0]; print(vid_name)
      BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
      idx = get_BBOX_IDX(BoxDims,vid_name)
      ArenaDims = BoxDims.iloc[idx,:].to_dict()

      # load the h5
      h5_obj = h5py.File(load_path + exp, "r")

      h5tracks = h5_obj['tracks'][:].T
      frame_num, _, _, track_num = h5tracks.shape
      print("frames x nodes x 2 x animals", h5tracks.shape)

      # load data from h5 tracks
      fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

      # get time stamps
      time_stamps = getTimeStamps(load_path + vid_name + '.csv')

      # plot all trajectories
      plt.figure(figsize=[3,3])
      for i in fish_data:
        byx = i.__getattribute__('mbx')
        byy = i.__getattribute__('mby')
        plt.scatter(byx,byy, s=3, alpha=0.1)
        plt.show()

      # # organize them into group arrays and save as lev0_basics.npz
      f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
      os.makedirs(output_path+exp, exist_ok=True)
      np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
              time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
              f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
              f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
              f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
              f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition)

    # break

# Experimental list (CL_num)

In [8]:
gsheet = 'https://docs.google.com/spreadsheets/d/1bSfcczIO4V62vPDB93Xjm2VDwlayalswnmi6PDiiLps/edit?gid=0#gid=0'
tab_name = 'CL_num'

experiment_class = f'Social escape - {tab_name}'
path = account_path + 'Yu_escape_paper/Behavior/' + experiment_class

data_path = path + '/Data/'
save_path = path + '/Analysis/'

metadata = read_metadata(gsheet,tab_name)
experiments = metadata.index

,experiment,time_stamps,tank,metadata,fish_num,sex,exp_type,fps_Hz,rig_mm,vid_px,body_length,experiment_date,condition
0,vid_1.mp4.000_vid_1.analysis.h5,vid_1.csv,T39,vid_1_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5
1,vid_9.mp4.000_vid_9.analysis.h5,vid_9.csv,T39,vid_9_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5
2,vid_8.mp4.000_vid_8.analysis.h5,vid_8.csv,T39,vid_8_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5
3,vid_7.mp4.000_vid_7.analysis.h5,vid_7.csv,T39,vid_7_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5
4,vid_6.mp4.000_vid_6.analysis.h5,vid_6.csv,T39,vid_6_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5
5,vid_5.mp4.000_vid_5.analysis.h5,vid_5.csv,T39,vid_5_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5
6,vid_4.mp4.000_vid_4.analysis.h5,vid_4.csv,T39,vid_4_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5
7,vid_3.mp4.000_vid_3.analysis.h5,vid_3.csv,T39,vid_3_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5
8,vid_2.mp4.000_vid_2.analysis.h5,vid_2.csv,T39,vid_2_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5
9,vid_11.mp4.000_vid_11.analysis.h5,vid_11.csv,T39,vid_11_metadata.csv,1,?,social_escape_CL_alternating,50,265,640,12,40425,5


# process

In [ ]:
for exp in experiments:
  if type(metadata.loc[exp]) ==pd.Series: # only 1 unique index
    load_path = data_path + f'{metadata.experiment_date[exp]}/'
    print(load_path)
    output_path = save_path + f'{metadata.experiment_date[exp]}/'
    print(output_path)

    os.makedirs(output_path, exist_ok=True)

    print(exp)
    exp_name = exp.split('.')[0]

    # load metadata
    exp_type = metadata.exp_type[exp]
    fish_num = metadata.fish_num[exp]
    fps_Hz = metadata.fps_Hz[exp]
    rig_mm = metadata.rig_mm[exp]
    vid_px = metadata.vid_px[exp]
    condition = metadata.condition[exp]
    scale = rig_mm / vid_px # mm/pixel

    # get arena dimensions
    vid_name = exp.split('.')[0]; print(vid_name)
    BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
    idx = get_BBOX_IDX(BoxDims,vid_name)
    ArenaDims = BoxDims.iloc[idx,:].to_dict()

    # get metdata for vid
    vid_metadata = pd.read_csv(load_path+exp_name+'_metadata.csv')
    vid_metadata = vid_metadata.set_index("Parameter")["Value"].to_dict()

    # load the h5
    h5_obj = h5py.File(load_path + exp, "r")

    h5tracks = h5_obj['tracks'][:].T
    frame_num, _, _, track_num = h5tracks.shape
    print("frames x nodes x 2 x animals", h5tracks.shape)

    # load data from h5 tracks
    fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

    # get time stamps
    time_stamps = getTimeStamps(load_path + vid_name + '.csv')

    # plot all trajectories
    plt.figure(figsize=[3,3])
    for i in fish_data:
      byx = i.__getattribute__('mbx')
      byy = i.__getattribute__('mby')
      plt.scatter(byx,byy, s=3, alpha=0.1)
      plt.show()

    # # organize them into group arrays and save as lev0_basics.npz
    f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
    os.makedirs(output_path+exp, exist_ok=True)
    np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
            time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
            f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
            f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
            f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
            f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition,vid_metadata=vid_metadata)

  else:
    for i in range(len(metadata.loc[exp])):
      load_path = data_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(load_path)
      output_path = save_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(output_path)

      os.makedirs(output_path, exist_ok=True)

      print(exp)
      exp_name = exp.split('.')[0]

      # load metadata
      exp_type = metadata.exp_type[exp].iloc[i]
      fish_num = metadata.fish_num[exp].iloc[i]
      fps_Hz = metadata.fps_Hz[exp].iloc[i]
      rig_mm = metadata.rig_mm[exp].iloc[i]
      vid_px = metadata.vid_px[exp].iloc[i]
      condition = metadata.condition[exp].iloc[i]
      scale = rig_mm / vid_px # mm/pixel

      # get arena dimensions
      vid_name = exp.split('.')[0]; print(vid_name)
      BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
      idx = get_BBOX_IDX(BoxDims,vid_name)
      ArenaDims = BoxDims.iloc[idx,:].to_dict()

      # get metdata for vid
      vid_metadata = pd.read_csv(load_path+exp_name+'_metadata.csv')
      vid_metadata = vid_metadata.set_index("Parameter")["Value"].to_dict()

      # load the h5
      h5_obj = h5py.File(load_path + exp, "r")

      h5tracks = h5_obj['tracks'][:].T
      frame_num, _, _, track_num = h5tracks.shape
      print("frames x nodes x 2 x animals", h5tracks.shape)

      # load data from h5 tracks
      fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

      # get time stamps
      time_stamps = getTimeStamps(load_path + vid_name + '.csv')

      # plot all trajectories
      plt.figure(figsize=[3,3])
      for i in fish_data:
        byx = i.__getattribute__('mbx')
        byy = i.__getattribute__('mby')
        plt.scatter(byx,byy, s=3, alpha=0.1)
        plt.show()

      # # organize them into group arrays and save as lev0_basics.npz
      f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
      os.makedirs(output_path+exp, exist_ok=True)
      np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
              time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
              f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
              f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
              f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
              f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition,vid_metadata=vid_metadata)

    # break

# Experimental list (BG_linear_escape)

In [9]:
gsheet = 'https://docs.google.com/spreadsheets/d/1bSfcczIO4V62vPDB93Xjm2VDwlayalswnmi6PDiiLps/edit?gid=0#gid=0'
tab_name = 'BG_linear_escape'

experiment_class = f'Social escape - {tab_name}'
path = account_path + 'Yu_escape_paper/Behavior/' + experiment_class

data_path = path + '/Data/'
save_path = path + '/Analysis/'

metadata = read_metadata(gsheet,tab_name)
experiments = metadata.index

,experiment,time_stamps,tank,metadata,fish_num,sex,exp_type,fps_Hz,rig_mm,vid_px,body_length,experiment_date,condition
0,vid_1.mp4.000_vid_1.analysis.h5,vid_1.csv,T61,vid_1_metadata.csv,1,f,DC_disappear_CL_alternating_linear_revision.py,50,265,800,12,21026,linear_disappear
1,vid_8.mp4.000_vid_8.analysis.h5,vid_8.csv,T61,vid_8_metadata.csv,1,f,DC_disappear_CL_alternating_linear_revision.py,50,265,800,12,21026,linear_disappear
2,vid_6.mp4.000_vid_6.analysis.h5,vid_6.csv,T61,vid_6_metadata.csv,1,m,DC_disappear_CL_alternating_linear_revision.py,50,265,800,12,21026,linear_disappear
3,vid_4.mp4.000_vid_4.analysis.h5,vid_4.csv,T61,vid_4_metadata.csv,1,m,DC_disappear_CL_alternating_linear_revision.py,50,265,800,12,21026,linear_disappear
4,vid_3.mp4.000_vid_3.analysis.h5,vid_3.csv,T61,vid_3_metadata.csv,1,m,DC_disappear_CL_alternating_linear_revision.py,50,265,800,12,21026,linear_disappear
...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,vid_14.mp4.000_vid_14.analysis.h5,vid_14.csv,U2,vid_14_metadata.csv,1,,DC_negative_CL_alternating.py,50,265,800,12,40726B,BG_negative
265,vid_13.mp4.000_vid_13.analysis.h5,vid_13.csv,U2,vid_13_metadata.csv,1,,DC_negative_CL_alternating.py,50,265,800,12,40726B,BG_negative
266,vid_12.mp4.000_vid_12.analysis.h5,vid_12.csv,U2,vid_12_metadata.csv,1,,DC_negative_CL_alternating.py,50,265,800,12,40726B,BG_negative
267,vid_11.mp4.000_vid_11.analysis.h5,vid_11.csv,U2,vid_11_metadata.csv,1,,DC_negative_CL_alternating.py,50,265,800,12,40726B,BG_negative


# process

In [ ]:
for exp in experiments:
  if type(metadata.loc[exp]) ==pd.Series: # only 1 unique index
    load_path = data_path + f'{metadata.experiment_date[exp]}/'
    print(load_path)
    output_path = save_path + f'{metadata.experiment_date[exp]}/'
    print(output_path)

    os.makedirs(output_path, exist_ok=True)

    print(exp)
    exp_name = exp.split('.')[0]

    # load metadata
    exp_type = metadata.exp_type[exp]
    fish_num = metadata.fish_num[exp]
    fps_Hz = metadata.fps_Hz[exp]
    rig_mm = metadata.rig_mm[exp]
    vid_px = metadata.vid_px[exp]
    condition = metadata.condition[exp]
    scale = rig_mm / vid_px # mm/pixel

    # get arena dimensions
    vid_name = exp.split('.')[0]; print(vid_name)
    BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
    idx = get_BBOX_IDX(BoxDims,vid_name)
    ArenaDims = BoxDims.iloc[idx,:].to_dict()

    # get metdata for vid
    vid_metadata = pd.read_csv(load_path+exp_name+'_metadata.csv')
    vid_metadata = vid_metadata.set_index("Parameter")["Value"].to_dict()

    # load the h5
    h5_obj = h5py.File(load_path + exp, "r")

    h5tracks = h5_obj['tracks'][:].T
    frame_num, _, _, track_num = h5tracks.shape
    print("frames x nodes x 2 x animals", h5tracks.shape)

    # load data from h5 tracks
    fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

    # get time stamps
    time_stamps = getTimeStamps(load_path + vid_name + '.csv')

    # plot all trajectories
    plt.figure(figsize=[3,3])
    for i in fish_data:
      byx = i.__getattribute__('mbx')
      byy = i.__getattribute__('mby')
      plt.scatter(byx,byy, s=3, alpha=0.1)
      plt.show()

    # # organize them into group arrays and save as lev0_basics.npz
    f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
    os.makedirs(output_path+exp, exist_ok=True)
    np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
            time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
            f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
            f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
            f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
            f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition,vid_metadata=vid_metadata)

  else:
    for i in range(len(metadata.loc[exp])):
      load_path = data_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(load_path)
      output_path = save_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(output_path)

      os.makedirs(output_path, exist_ok=True)

      print(exp)
      exp_name = exp.split('.')[0]

      # load metadata
      exp_type = metadata.exp_type[exp].iloc[i]
      fish_num = metadata.fish_num[exp].iloc[i]
      fps_Hz = metadata.fps_Hz[exp].iloc[i]
      rig_mm = metadata.rig_mm[exp].iloc[i]
      vid_px = metadata.vid_px[exp].iloc[i]
      condition = metadata.condition[exp].iloc[i]
      scale = rig_mm / vid_px # mm/pixel

      # get arena dimensions
      vid_name = exp.split('.')[0]; print(vid_name)
      BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
      idx = get_BBOX_IDX(BoxDims,vid_name)
      ArenaDims = BoxDims.iloc[idx,:].to_dict()

      # get metdata for vid
      vid_metadata = pd.read_csv(load_path+exp_name+'_metadata.csv')
      vid_metadata = vid_metadata.set_index("Parameter")["Value"].to_dict()

      # load the h5
      h5_obj = h5py.File(load_path + exp, "r")

      h5tracks = h5_obj['tracks'][:].T
      frame_num, _, _, track_num = h5tracks.shape
      print("frames x nodes x 2 x animals", h5tracks.shape)

      # load data from h5 tracks
      fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

      # get time stamps
      time_stamps = getTimeStamps(load_path + vid_name + '.csv')

      # plot all trajectories
      plt.figure(figsize=[3,3])
      for i in fish_data:
        byx = i.__getattribute__('mbx')
        byy = i.__getattribute__('mby')
        plt.scatter(byx,byy, s=3, alpha=0.1)
        plt.show()

      # # organize them into group arrays and save as lev0_basics.npz
      f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
      os.makedirs(output_path+exp, exist_ok=True)
      np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
              time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
              f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
              f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
              f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
              f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition,vid_metadata=vid_metadata)

    # break

# Experimental list (Orb_stim)

In [10]:
gsheet = 'https://docs.google.com/spreadsheets/d/1bSfcczIO4V62vPDB93Xjm2VDwlayalswnmi6PDiiLps/edit?gid=0#gid=0'
tab_name = 'Orb_stim'

experiment_class = f'Social escape - {tab_name}'
path = account_path + 'Yu_escape_paper/Behavior/' + experiment_class

data_path = path + '/Data/'
save_path = path + '/Analysis/'

metadata = read_metadata(gsheet,tab_name)
experiments = metadata.index

,experiment,time_stamps,tank,metadata,fish_num,sex,exp_type,fps_Hz,rig_mm,vid_px,body_length,experiment_date,condition
0,vid_9.mp4.000_vid_9.analysis.h5,vid_9.csv,F3,NA,1,m,3 trial OL escape (DC_escape.py),50,265,800,12,12226,orb_stim
1,vid_1.mp4.000_vid_1.analysis.h5,vid_1.csv,F3,NA,1,f,3 trial OL escape (DC_escape.py),50,265,800,12,12226,orb_stim
2,vid_8.mp4.000_vid_8.analysis.h5,vid_8.csv,F3,NA,1,m,3 trial OL escape (DC_escape.py),50,265,800,12,12226,orb_stim
3,vid_7.mp4.000_vid_7.analysis.h5,vid_7.csv,F3,NA,1,m,3 trial OL escape (DC_escape.py),50,265,800,12,12226,orb_stim
4,vid_6.mp4.000_vid_6.analysis.h5,vid_6.csv,F3,NA,1,f,3 trial OL escape (DC_escape.py),50,265,800,12,12226,orb_stim
5,vid_5.mp4.000_vid_5.analysis.h5,vid_5.csv,F3,NA,1,m,3 trial OL escape (DC_escape.py),50,265,800,12,12226,orb_stim
6,vid_4.mp4.000_vid_4.analysis.h5,vid_4.csv,F3,NA,1,f,3 trial OL escape (DC_escape.py),50,265,800,12,12226,orb_stim
7,vid_3.mp4.000_vid_3.analysis.h5,vid_3.csv,F3,NA,1,m,3 trial OL escape (DC_escape.py),50,265,800,12,12226,orb_stim
8,vid_2.mp4.000_vid_2.analysis.h5,vid_2.csv,F3,NA,1,m,3 trial OL escape (DC_escape.py),50,265,800,12,12226,orb_stim
9,vid_6.mp4.000_vid_6.analysis.h5,vid_6.csv,T61,NA,1,f,3 trial OL escape (DC_escape.py),50,265,800,12,12626,orb_stim_small_grey


# process

In [ ]:
for exp in experiments:
  if type(metadata.loc[exp]) ==pd.Series: # only 1 unique index
    load_path = data_path + f'{metadata.experiment_date[exp]}/'
    print(load_path)
    output_path = save_path + f'{metadata.experiment_date[exp]}/'
    print(output_path)

    os.makedirs(output_path, exist_ok=True)

    print(exp)
    exp_name = exp.split('.')[0]

    # load metadata
    exp_type = metadata.exp_type[exp]
    fish_num = metadata.fish_num[exp]
    fps_Hz = metadata.fps_Hz[exp]
    rig_mm = metadata.rig_mm[exp]
    vid_px = metadata.vid_px[exp]
    condition = metadata.condition[exp]
    scale = rig_mm / vid_px # mm/pixel

    # get arena dimensions
    vid_name = exp.split('.')[0]; print(vid_name)
    BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
    idx = get_BBOX_IDX(BoxDims,vid_name)
    ArenaDims = BoxDims.iloc[idx,:].to_dict()


    # load the h5
    h5_obj = h5py.File(load_path + exp, "r")

    h5tracks = h5_obj['tracks'][:].T
    frame_num, _, _, track_num = h5tracks.shape
    print("frames x nodes x 2 x animals", h5tracks.shape)

    # load data from h5 tracks
    fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

    # get time stamps
    time_stamps = getTimeStamps(load_path + vid_name + '.csv')

    # plot all trajectories
    plt.figure(figsize=[3,3])
    for i in fish_data:
      byx = i.__getattribute__('mbx')
      byy = i.__getattribute__('mby')
      plt.scatter(byx,byy, s=3, alpha=0.1)
      plt.show()

    # # organize them into group arrays and save as lev0_basics.npz
    f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
    os.makedirs(output_path+exp, exist_ok=True)
    np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
            time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
            f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
            f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
            f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
            f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition)

  else:
    for i in range(len(metadata.loc[exp])):
      load_path = data_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(load_path)
      output_path = save_path + f'{metadata.experiment_date[exp].iloc[i]}/'
      print(output_path)

      os.makedirs(output_path, exist_ok=True)

      print(exp)
      exp_name = exp.split('.')[0]

      # load metadata
      exp_type = metadata.exp_type[exp].iloc[i]
      fish_num = metadata.fish_num[exp].iloc[i]
      fps_Hz = metadata.fps_Hz[exp].iloc[i]
      rig_mm = metadata.rig_mm[exp].iloc[i]
      vid_px = metadata.vid_px[exp].iloc[i]
      condition = metadata.condition[exp].iloc[i]
      scale = rig_mm / vid_px # mm/pixel

      # get arena dimensions
      vid_name = exp.split('.')[0]; print(vid_name)
      BoxDims = pd.read_csv(load_path + 'bounding_box_four_corners.csv')
      idx = get_BBOX_IDX(BoxDims,vid_name)
      ArenaDims = BoxDims.iloc[idx,:].to_dict()

      # load the h5
      h5_obj = h5py.File(load_path + exp, "r")

      h5tracks = h5_obj['tracks'][:].T
      frame_num, _, _, track_num = h5tracks.shape
      print("frames x nodes x 2 x animals", h5tracks.shape)

      # load data from h5 tracks
      fish_data = process_h5(h5tracks,fish_num,sleap_nodes,exp)

      # get time stamps
      time_stamps = getTimeStamps(load_path + vid_name + '.csv')

      # plot all trajectories
      plt.figure(figsize=[3,3])
      for i in fish_data:
        byx = i.__getattribute__('mbx')
        byy = i.__getattribute__('mby')
        plt.scatter(byx,byy, s=3, alpha=0.1)
        plt.show()

      # # organize them into group arrays and save as lev0_basics.npz
      f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed, f_disp_heading, f_lex, f_ley, f_rex, f_rey = group_arrays(fish_data,fish_num,frame_num,fps_Hz,scale)
      os.makedirs(output_path+exp, exist_ok=True)
      np.savez(output_path+exp+'/lev0_basics.npz', exp_type=exp_type, fish_num=fish_num, ArenaDims=ArenaDims, scale=scale, fps=fps_Hz, \
              time_stamps=time_stamps['elapsed_seconds'], fish_showing=time_stamps['fish_showing'], fish_escaping=time_stamps['fish_escaping'], \
              f_bodylength_mm=f_bodylength_mm, f_bodylength_px=f_bodylength_px, f_nosex=f_nosex, f_nosey=f_nosey, \
              f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily, \
              f_disp_heading=f_disp_heading, f_lex=f_lex, f_ley=f_ley, f_rex=f_rex, f_rey=f_rey, \
              f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed, condition=condition)

    # break